# Notebook 1 — Data Loading, Validation, Aggregation and ML Table Construction

## 1. Import Libraries and Database Configuration

In [4]:
import pandas as pd
import sqlalchemy

print("Pandas version:", pd.__version__)
print("SQLAlchemy version:", sqlalchemy.__version__)

Pandas version: 3.0.3
SQLAlchemy version: 2.0.52


## 2. Database Connection

In [5]:
from sqlalchemy import create_engine

DB_USER = "postgres"
DB_PASSWORD = input("Enter PostgreSQL password: ")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "olist_db"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected successfully!")

Connected successfully!


## 3. Loading Data from PostgreSQL

In [6]:
orders = pd.read_sql("SELECT * FROM orders", engine)
customers = pd.read_sql("SELECT * FROM customers", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
order_payments = pd.read_sql("SELECT * FROM order_payments", engine)
products = pd.read_sql("SELECT * FROM products", engine)
sellers = pd.read_sql("SELECT * FROM sellers", engine)

print("orders:", orders.shape)
print("customers:", customers.shape)
print("order_items:", order_items.shape)
print("order_payments:", order_payments.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)

orders: (99441, 8)
customers: (99441, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
products: (32951, 9)
sellers: (3095, 4)


## 4. Column Inspection

In [7]:
print("ORDERS")
print(orders.columns.tolist())

print("\nCUSTOMERS")
print(customers.columns.tolist())

print("\nORDER_ITEMS")
print(order_items.columns.tolist())

print("\nORDER_PAYMENTS")
print(order_payments.columns.tolist())

print("\nPRODUCTS")
print(products.columns.tolist())

print("\nSELLERS")
print(sellers.columns.tolist())

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

ORDER_PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

SELLERS
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


## 5. Order-Level Statistics

In [8]:
print("Orders:", orders["order_id"].nunique(), "unique orders")

print("Order items:", order_items["order_id"].nunique(), "unique orders")

print("Order payments:", order_payments["order_id"].nunique(), "unique orders")

Orders: 99441 unique orders
Order items: 98666 unique orders
Order payments: 99440 unique orders


In [9]:
print("Average items per order:",
      len(order_items) / order_items["order_id"].nunique())

print("Average payment rows per order:",
      len(order_payments) / order_payments["order_id"].nunique())

Average items per order: 1.1417306873695092
Average payment rows per order: 1.0447103781174578


## 6. Duplicate Checks

In [10]:
print("Duplicate order IDs in orders:",
      orders["order_id"].duplicated().sum())

print("Duplicate customer IDs in customers:",
      customers["customer_id"].duplicated().sum())
    
print("Duplicate product IDs in products:",
      products["product_id"].duplicated().sum())

print("Duplicate seller IDs in sellers:",
      sellers["seller_id"].duplicated().sum())

Duplicate order IDs in orders: 0
Duplicate customer IDs in customers: 0
Duplicate product IDs in products: 0
Duplicate seller IDs in sellers: 0


## 7. Missing Values Check

In [12]:
print("Missing values in orders:")
print(orders.isna().sum())

print("\nMissing values in customers:")
print(customers.isna().sum())

print("\nMissing values in order_items:")
print(order_items.isna().sum())

print("\nMissing values in order_payments:")
print(order_payments.isna().sum())

Missing values in orders:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Missing values in customers:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Missing values in order_items:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Missing values in order_payments:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64


## 8. Order Items Aggregation

In [14]:
items_agg = (
    order_items.groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

items_agg.head()

,order_id,total_items,total_price,total_freight,unique_products,unique_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,1,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,1,1


In [15]:
print(items_agg.shape)
print(items_agg["order_id"].nunique())

(98666, 6)
98666


## 9. Payment Aggregation

In [16]:
payments_agg = (
    order_payments.groupby("order_id")
    .agg(
        total_payments=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        payment_types=("payment_type", "nunique")
    )
    .reset_index()
)

payments_agg.head()

,order_id,total_payments,payment_count,payment_types
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1


In [17]:
print(payments_agg.shape)
print(payments_agg["order_id"].nunique())

(99440, 4)
99440


## 10. Building the ML Table

In [19]:
ml_table = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="1:1"
)

print("Shape:", ml_table.shape)
print("Unique orders:", ml_table["order_id"].nunique())

Shape: (99441, 12)
Unique orders: 99441


In [20]:
ml_table = ml_table.merge(
    items_agg,
    on="order_id",
    how="left",
    validate="1:1"
)

ml_table = ml_table.merge(
    payments_agg,
    on="order_id",
    how="left",
    validate="1:1"
)

print("Shape:", ml_table.shape)
print("Unique orders:", ml_table["order_id"].nunique())

Shape: (99441, 20)
Unique orders: 99441


## 11. Final ML Table Validation

In [21]:
print("Final shape:", ml_table.shape)

print("Total rows:", len(ml_table))

print("Unique order IDs:", ml_table["order_id"].nunique())

print("Duplicate order IDs:",
      ml_table["order_id"].duplicated().sum())

Final shape: (99441, 20)
Total rows: 99441
Unique order IDs: 99441
Duplicate order IDs: 0


## 12. Missing Values After Merging

In [22]:
ml_table.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
total_items                       775
total_price                       775
total_freight                     775
unique_products                   775
unique_sellers                    775
total_payments                      1
payment_count                       1
payment_types                       1
dtype: int64

## 13. Saving the ML Table

In [23]:
from pathlib import Path

output_dir = Path("../data/artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "ml_table.csv"

ml_table.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(ml_table)}")
print(f"Columns: {len(ml_table.columns)}")

Saved: ..\data\artifacts\ml_table.csv
Rows: 99441
Columns: 20
